# Lab 10: From Exploratory to Explanatory: Annotating Charts in Matplotlib

## Goal for today and submission
This is a **light lab** to help with your Project 2 final piece and the static charts in your notebook's About section. Work through it in class, then use the rest of your time on your narrative structure check-in (due Sunday). Submit this notebook as HTML on Canvas by Sunday.

## Objectives

By the end of this lab, you should be able to:

- Turn a default, exploratory chart into an explanatory one with five moves: an **active title**, **decluttering**, **highlighting**, **direct labels**, and **annotation**
- Place text and arrows with `ax.text()` and `ax.annotate()`, in data coordinates or axes coordinates
- Export a chart in a format you can assemble into an infographic or science figure

## Setup

We'll use the Titanic passenger list and the gapminder data (built into Plotly).

In [ ]:
# !pip install pandas matplotlib plotly

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px   # only for its built-in gapminder data

titanic = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")
gap = px.data.gapminder()

# Survival rate for each class and sex
rates = titanic.groupby(["class", "sex"])["survived"].mean().unstack()
rates

## 1. The default chart

Here is the chart you'd make while exploring. It's accurate, and it's fine for you. But it doesn't tell a reader what to look at.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
rates.plot(kind="bar", ax=ax)
ax.set_title("Survival rate by class and sex")
plt.show()

<div class="alert alert-warning">
<b>Stop and think:</b>

What is the **one thing** you'd want a reader to take away from this chart? Now list everything on the chart that doesn't help them see it.
</div>

## 2. Five moves from exploratory to explanatory

| Move | What it does | Matplotlib tools |
|---|---|---|
| 1. **Active title** | States the takeaway, not the variables. A subtitle says what's plotted. | `ax.set_title(..., loc="left")`, `ax.text(..., transform=ax.transAxes)` |
| 2. **Declutter** | Removes ink that doesn't carry information: boxes, extra gridlines, axes you've replaced with labels. | `ax.spines[...].set_visible(False)`, `ax.set_yticks([])`, `ax.tick_params(length=0)` |
| 3. **Highlight** | Color for what the story is about, grey for context. | `color=` on each bar or line |
| 4. **Direct labels** | Puts labels on the data instead of in a legend or on an axis the reader has to trace. | `ax.bar_label()`, `ax.text()` |
| 5. **Annotate** | Points at the one detail the reader shouldn't miss, in words. | `ax.annotate(text, xy=..., xytext=..., arrowprops=...)` |

Plus one habit: **cite your source** on the chart (`fig.text()`), which Project 2 requires.

### Two coordinate systems

When you place text, you have to say *where*:

- **Data coordinates** (the default): `x` and `y` in the units of your axes. Use these to point at a bar or a point, so the label moves with the data.
- **Axes coordinates** (`transform=ax.transAxes`): `(0, 0)` is the bottom-left of the plot area and `(1, 1)` the top-right. Use these for things tied to the frame, like a subtitle.

In `ax.annotate()`, `xy` is where the arrow **points** and `xytext` is where the **text** sits.

## 3. The makeover

Here's the same data with all five moves applied. Read the comments to see which line does which move, and fill in the TODOs.

In [ ]:
classes = ["First", "Second", "Third"]
x = np.arange(len(classes))   # bar positions: 0, 1, 2
w = 0.38                      # bar width
HIGHLIGHT, MUTED = "#0072B2", "#bbbbbb"

fig, ax = plt.subplots(figsize=(7, 4.2))

# Move 3, highlight: the group the story is about gets color; the comparison group is grey
women = ax.bar(x - w/2, rates.loc[classes, "female"], w, color=____  # TODO)
men = ax.bar(x + w/2, rates.loc[classes, "male"], w, color=____  # TODO)

# Move 2, declutter: no box, no y-axis (the bars will be labeled directly), no tick marks
for side in [____  # TODO: which spines?]:
    ax.spines[side].set_visible(False)
ax.set_yticks([])
ax.set_xticks(x, [f"{c} class" for c in classes])
ax.tick_params(axis="x", length=0)
ax.set_ylim(0, 1.1)

# Move 4, direct labels: values on the bars, and group names inside the first pair (no legend)
ax.bar_label(women, labels=[f"{v:.0%}" for v in women.datavalues], padding=3, color=HIGHLIGHT, fontweight="bold")
ax.bar_label(men, labels=[f"{v:.0%}" for v in men.datavalues], padding=3, color="#666666")
ax.text(x[0] - w/2, 0.04, "Women", ha="center", color="white", fontweight="bold", fontsize=9)
ax.text(x[0] + w/2, 0.04, "Men", ha="center", color="#333333", fontweight="bold", fontsize=9)

# Move 5, annotate: point at the one detail the reader shouldn't miss
ax.annotate("____"  # TODO: write the annotation text,
            xy=(x[2] - w/2, rates.loc["Third", "female"] + 0.08),   # the arrow tip (data coordinates)
            xytext=(1.45, 0.82),                                    # where the text sits
            arrowprops=dict(arrowstyle="->", color="#333333"), fontsize=9, color="#333333")

# Move 1, active title + subtitle: the title states the takeaway; the subtitle says what's plotted
ax.set_title("____"  # TODO: write an active title that states the takeaway,
             loc="left", fontweight="bold", fontsize=12, pad=22)
ax.text(0, 1.03, "Share of passengers who survived, by ticket class and sex",
        transform=ax.transAxes, color="#555555", fontsize=10)

# Source line, bottom left
fig.text(0.01, -0.03, "Source: Titanic passenger list (seaborn-data), n = 891 passengers",
         fontsize=8, color="#777777")
fig.tight_layout()
plt.show()

<div class="alert alert-info">
<b>Talk to a neighbor:</b>

1. Why are men grey instead of a second bright color? When *would* you give both groups strong colors?
2. We deleted the y-axis entirely. What replaced it, and when would deleting the y-axis be a bad idea?
3. Cover the title and read only the chart. Does it still make the same point? Now cover the chart and read only the title. Would a reader who only skims the title get the takeaway?
</div>

## 4. Your turn: a line chart

The same moves work on line charts. The data: median life expectancy across the countries in each continent, every five years. The story: **Asia's life expectancy rose by 27 years and caught up with the Americas.**

Make Asia the highlight, label each line directly at its right end (instead of a legend), and add an annotation, an active title and subtitle, and a source line.

In [ ]:
med = gap.groupby(["continent", "year"], as_index=False)["lifeExp"].median()

fig, ax = plt.subplots(figsize=(7, 4.2))
for continent, grp in med.groupby("continent"):
    # TODO (highlight): make Asia blue and thick, every other continent thin and grey
    ax.plot(grp["year"], grp["lifeExp"])

    # TODO (direct labels): label each line at its last point with ax.text(), in the line's color

# TODO (declutter): remove the top and right spines

# TODO (annotate): point at Asia's line and say how much it rose (44.9 years in 1952 -> 72.4 in 2007)

# TODO (active title + subtitle + source)

fig.tight_layout()
plt.show()

<div class="alert alert-warning">
<b>Stop and think:</b>

Direct labels at the ends of lines can collide when two lines finish close together, as Asia and the Americas do here. How did the solution handle it? What else could you do?
</div>

## 5. Exporting for your final piece

You're allowed to assemble your Project 2 infographic or science figure in another tool (Figma, Canva, Illustrator, Inkscape, PowerPoint), but every data mark has to come from your code. Export each chart like this:

- **SVG or PDF** (vector): stays sharp at any size, and in Figma, Illustrator, or Inkscape you can still edit the text. Best for science figures.
- **PNG at 300 dpi**: works everywhere, including Canva and PowerPoint.
- `bbox_inches="tight"` trims the extra white space, and `transparent=True` drops the white background so the chart sits on your design's background.

In [ ]:
fig.savefig("asia_life_expectancy.svg", bbox_inches="tight")
fig.savefig("asia_life_expectancy.png", dpi=300, bbox_inches="tight", transparent=True)

## On your own

Pick a static chart you plan to use in your **Project 2** notebook's About section or your final piece. (If you don't have one yet, use any chart from Lab 6 or your Project 1 report.)

1. Make the default version first.
2. Apply **at least three** of the five moves, plus a source line.
3. In the markdown cell below, write 2–3 sentences: what's the takeaway, which moves did you use, and why those?

In [ ]:
# TODO: your default chart



In [ ]:
# TODO: your explanatory version



**Type your 2–3 sentences here**

## Submission

Export this notebook to HTML (`Jupyter: Export to HTML` from the Command Palette, or the "..." menu in the notebook toolbar) and submit the HTML file on Canvas.